In [1]:
import pandas as pd
from tqdm import tqdm
from functools import partial
import joblib
import numpy as np
import faiss
import json
from sentence_transformers import SentenceTransformer
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from src.data.preprocessing import clean_text
from src.data.vectorizer import get_vectorizer
from src.models.tfidf_logreg import get_model_v01
from src.models.tfidf_svm import get_model_v01 as get_svm_model_v01
from src.evaluation.metrics import evaluate, format_cm
from src.data.rac_utils import retrive_related, retrive_related_embedding
from src.data.retrieval_features import agument_data_optimized
from src.models.tfidf_xgboost import get_xgboost_model_v02


c:\Work\Project\ticket-nlp-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "../data/raw/all_tickets_processed_improved_v3.csv"
df = pd.read_csv(DATA_PATH)

X = df["Document"]
y = df["Topic_group"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=2)

vectorizer: TfidfVectorizer = joblib.load("../artifacts/tfidf_vectorizer_v01.pkl")
lr_model: LogisticRegression = joblib.load("../artifacts/logreg_model_v01.pkl")
svm_model: LinearSVC = joblib.load("../artifacts/svm_model_v01.pkl")
xgboost_model: xgb.XGBClassifier = joblib.load("../artifacts/xgboost_v01.pkl")

with open("../artifacts/rac_corpus_similarity-euclidian_index_v01.json", 'r') as f:
    corpus = json.load(f)

index = faiss.read_index("../artifacts/traindata_similarity_index_v01.index")

retrieval_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

labelencoder: LabelEncoder = joblib.load("../artifacts/labelencoder_neural_v01.pkl")

In [3]:
X_train_agumented = await agument_data_optimized(
    texts=X_train.to_list(),
    vectorizer=vectorizer, retrieval_model=retrieval_model,
    index_path="../artifacts/traindata_similarity_index_v01.index",
    index_count=10,
    corpus=corpus, keys=["Document", "Topic_group"],
    y_key="Topic_group",
    classes=labelencoder.classes_,
    k=12,
    remove_top=False
)
X_train_agumented.shape

100%|██████████| 3827/3827 [02:07<00:00, 29.90it/s]


(38269, 30010)

In [4]:
X_test_agumented = await agument_data_optimized(
    texts=X_test.to_list(),
    vectorizer=vectorizer, retrieval_model=retrieval_model,
    index_path="../artifacts/traindata_similarity_index_v01.index",
    index_count=10,
    corpus=corpus, keys=["Document", "Topic_group"],
    y_key="Topic_group",
    classes=labelencoder.classes_,
    k=12,
    remove_top=False
)
X_test_agumented.shape

100%|██████████| 957/957 [00:31<00:00, 30.63it/s]


(9568, 30010)

In [5]:
y_train_encoded = labelencoder.transform(y_train)
y_test_encoded = labelencoder.transform(y_test)

In [6]:
model = get_xgboost_model_v02(len(labelencoder.classes_))

In [7]:
model.fit(
    X_train_agumented,
    y_train_encoded,
    eval_set = [(X_test_agumented, y_test_encoded)],
    verbose = True
)

[0]	validation_0-mlogloss:1.88698
[1]	validation_0-mlogloss:1.76925
[2]	validation_0-mlogloss:1.68439
[3]	validation_0-mlogloss:1.59514
[4]	validation_0-mlogloss:1.51671
[5]	validation_0-mlogloss:1.44616
[6]	validation_0-mlogloss:1.38785
[7]	validation_0-mlogloss:1.33091
[8]	validation_0-mlogloss:1.28062
[9]	validation_0-mlogloss:1.23457
[10]	validation_0-mlogloss:1.18985
[11]	validation_0-mlogloss:1.14989
[12]	validation_0-mlogloss:1.11106
[13]	validation_0-mlogloss:1.07759
[14]	validation_0-mlogloss:1.04363
[15]	validation_0-mlogloss:1.01221
[16]	validation_0-mlogloss:0.98347
[17]	validation_0-mlogloss:0.95635
[18]	validation_0-mlogloss:0.93015
[19]	validation_0-mlogloss:0.90623
[20]	validation_0-mlogloss:0.88368
[21]	validation_0-mlogloss:0.86226
[22]	validation_0-mlogloss:0.84264
[23]	validation_0-mlogloss:0.82422
[24]	validation_0-mlogloss:0.80709
[25]	validation_0-mlogloss:0.79078
[26]	validation_0-mlogloss:0.77534
[27]	validation_0-mlogloss:0.76074
[28]	validation_0-mlogloss:0.7

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [8]:
y_pred = model.predict(X_test_agumented)
format_cm(evaluate(y_test_encoded, y_pred), class_names=list(labelencoder.classes_), normalize=True)

              precision    recall  f1-score   support

           0     0.8966    0.8884    0.8925      1425
           1     0.8170    0.7358    0.7743       352
           2     0.8319    0.8639    0.8476      2183
           3     0.8325    0.8392    0.8358      2724
           4     0.8354    0.7783    0.8059       424
           5     0.8107    0.8038    0.8073      1412
           6     0.9153    0.8986    0.9069       493
           7     0.8867    0.8739    0.8802       555

    accuracy                         0.8455      9568
   macro avg     0.8533    0.8352    0.8438      9568
weighted avg     0.8457    0.8455    0.8454      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.888421,0.002105,0.035088,0.042807,0.004912,0.021053,0.002105,0.003509
True: Administrative rights,0.019886,0.735795,0.031250,0.178977,0.002841,0.014205,0.011364,0.005682
True: HR Support,0.014659,0.003665,0.863949,0.060467,0.009620,0.042602,0.000916,0.004123
True: Hardware,0.024229,0.011013,0.060206,0.839207,0.007709,0.038179,0.009912,0.009545
True: Internal Project,0.021226,0.004717,0.084906,0.049528,0.778302,0.049528,0.002358,0.009434
True: Miscellaneous,0.014873,0.006374,0.067989,0.084986,0.008499,0.803824,0.002833,0.010623
True: Purchase,0.006085,0.002028,0.018256,0.058824,0.006085,0.008114,0.898580,0.002028
True: Storage,0.014414,0.009009,0.027027,0.061261,0.000000,0.014414,0.000000,0.873874


In [9]:
from src.models.tfidf_logreg import get_model_v01
from src.models.tfidf_svm import get_model_v01 as get_svm_model_v01

In [14]:
lr_model = get_model_v01()
lr_model.fit(X_train_agumented, y_train_encoded)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [15]:
y_pred_lr = lr_model.predict(X_test_agumented)
format_cm(evaluate(y_test_encoded, y_pred_lr), class_names=list(labelencoder.classes_), normalize=True)

              precision    recall  f1-score   support

           0     0.9109    0.8891    0.8999      1425
           1     0.8489    0.7500    0.7964       352
           2     0.8648    0.8759    0.8703      2183
           3     0.8425    0.8796    0.8606      2724
           4     0.8597    0.7948    0.8260       424
           5     0.8320    0.8208    0.8264      1412
           6     0.9236    0.9067    0.9150       493
           7     0.9077    0.8865    0.8970       555

    accuracy                         0.8648      9568
   macro avg     0.8737    0.8504    0.8614      9568
weighted avg     0.8652    0.8648    0.8646      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.889123,0.002807,0.032982,0.044211,0.006316,0.019649,0.001404,0.003509
True: Administrative rights,0.019886,0.750000,0.025568,0.176136,0.002841,0.014205,0.008523,0.002841
True: HR Support,0.010536,0.001832,0.875859,0.059551,0.006413,0.038021,0.001832,0.005955
True: Hardware,0.019824,0.011380,0.042217,0.879589,0.004405,0.030470,0.006975,0.005140
True: Internal Project,0.021226,0.007075,0.066038,0.056604,0.794811,0.047170,0.002358,0.004717
True: Miscellaneous,0.014873,0.001416,0.055949,0.080737,0.010623,0.820822,0.005666,0.009915
True: Purchase,0.006085,0.002028,0.016227,0.048682,0.008114,0.010142,0.906694,0.002028
True: Storage,0.012613,0.003604,0.023423,0.055856,0.000000,0.018018,0.000000,0.886486


In [16]:
svm_model = get_svm_model_v01()
svm_model.fit(X_train_agumented, y_train_encoded)

y_pred_svm = svm_model.predict(X_test_agumented)

In [17]:
format_cm(evaluate(y_test_encoded, y_pred_svm), class_names=list(labelencoder.classes_), normalize=True)


              precision    recall  f1-score   support

           0     0.9181    0.9046    0.9113      1425
           1     0.8459    0.7642    0.8030       352
           2     0.8773    0.8814    0.8793      2183
           3     0.8554    0.8862    0.8705      2724
           4     0.8589    0.8184    0.8382       424
           5     0.8515    0.8407    0.8460      1412
           6     0.9143    0.9087    0.9115       493
           7     0.9208    0.9009    0.9107       555

    accuracy                         0.8756      9568
   macro avg     0.8803    0.8631    0.8713      9568
weighted avg     0.8758    0.8756    0.8755      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.904561,0.002807,0.026667,0.037895,0.004912,0.018246,0.001404,0.003509
True: Administrative rights,0.014205,0.764205,0.022727,0.161932,0.002841,0.014205,0.019886,0.000000
True: HR Support,0.010994,0.002290,0.881356,0.055428,0.006413,0.036647,0.002290,0.004581
True: Hardware,0.018355,0.010279,0.040749,0.886197,0.005140,0.026432,0.008076,0.004772
True: Internal Project,0.021226,0.007075,0.056604,0.054245,0.818396,0.033019,0.002358,0.007075
True: Miscellaneous,0.014164,0.003541,0.046742,0.071530,0.011331,0.840652,0.003541,0.008499
True: Purchase,0.004057,0.004057,0.018256,0.048682,0.010142,0.006085,0.908722,0.000000
True: Storage,0.009009,0.003604,0.023423,0.050450,0.000000,0.012613,0.000000,0.900901


In [27]:
set(y_train.to_list())

{'Access',
 'Administrative rights',
 'HR Support',
 'Hardware',
 'Internal Project',
 'Miscellaneous',
 'Purchase',
 'Storage'}

In [28]:
y_train.to_list()[0]

'Hardware'

In [29]:
X_train.to_list()[0]

'unable to connect to unable connect hi longer able connect was working earlier morning please can you urgently advise working active bid need update files share point can following logs warning used setting maximum version options error ca fails with such file or directory options error please correct these errors use help for more information client manager'